# This is a notebook to read databento databases

In [ ]:
import databento as db

In [ ]:
# Read saved .dbn.zst
zst_file_path = r"../data/oil-glbx-mdp3-20240311-20251110.ohlcv-1m.dbn.zst"
stored_data = db.DBNStore.from_file(zst_file_path)

# Convert to dataframe
df = stored_data.to_df()

In [ ]:
print(df.head(100))

In [ ]:
# 0) drop calendar spreads (ESZ1-ESH2, etc.)
df_processed = df[~df['symbol'].str.contains('-', na=False)].copy()

# припускаємо, що вже прибрали спреди та маєте index = ['ts_event','instrument_id']
tmp = df_processed.reset_index()
tmp['day'] = tmp['ts_event'].dt.tz_convert('UTC').dt.floor('D')

# 1) денні обсяги
daily_vol = (tmp.groupby(['day','instrument_id'])['volume']
               .sum()
               .reset_index())

# 2) фронт-інструмент на день (найбільший обсяг)
front = (daily_vol.sort_values(['day','volume'], ascending=[True, False])
                 .drop_duplicates(subset=['day'])
                 .rename(columns={'instrument_id':'front_instrument'})[['day','front_instrument']])

# 3) лишаємо на кожен день тільки фронт-місяць
picked = (tmp.merge(front, on='day', how='left')
            .loc[lambda x: x['instrument_id'] == x['front_instrument']]
            .drop(columns=['day','front_instrument'])
            .set_index('ts_event')
            .sort_index())

picked.to_csv(r"../data/oil-glbx-mdp3-20240311-20251110.ohlcv-1m.csv")

In [ ]:
# 0) drop calendar spreads (ESZ1-ESH2, etc.)
df_processed = df[~df['symbol'].str.contains('-', na=False)].copy()

# 1) make (ts_event, instrument_id) a MultiIndex
df_processed.index.name = 'ts_event'                  # name the index (optional but nice)

# 2) sort and drop duplicates on the index pair
df_processed = df_processed.sort_index()                        # sorts by ts_event then instrument_id
df_processed = df_processed[~df_processed.index.duplicated(keep='first')] # keep the earliest print per (ts_event, instrument)

# check
assert not df_processed.index.duplicated().any()

In [ ]:
df_processed.to_csv(r"../data/glbx-mdp3-20100606-20251110.ohlcv-1m_source.csv")